# 04 - Evaluación

Evaluación del modelo en validación propia + análisis de errores + experimento de inferencia.

In [ ]:
import json
import time
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Baseline: modelo sin adapter

In [ ]:
# Evaluar baseline (sin adapter)
print('=== Baseline: Modelo sin adapter ===')

# Modificar config para usar sin adapter
with open('config.json', 'r') as f:
    config = json.load(f)

config_baseline = config.copy()
config_baseline['usar_adapter'] = False

# TODO: Ejecutar evaluar.py con config baseline
print('Ejecutar: python evaluar.py con usar_adapter=false')

## 2. Evaluar Config A y B

In [ ]:
# Evaluar con adapter Config A
print('=== Evaluando Config A ===')
# TODO: python evaluar.py con adapter config_a

# Evaluar con adapter Config B
print('=== Evaluando Config B ===')
# TODO: python evaluar.py con adapter config_b

## 3. Comparar resultados

In [ ]:
def comparar_resultados(ruta_baseline, ruta_config_a, ruta_config_b):
    """Compara resultados de baseline vs configs."""
    baseline = pd.read_csv(ruta_baseline) if Path(ruta_baseline).exists() else None
    config_a = pd.read_csv(ruta_config_a) if Path(ruta_config_a).exists() else None
    config_b = pd.read_csv(ruta_config_b) if Path(ruta_config_b).exists() else None
    
    resultados = {}
    if baseline is not None:
        ex_base = baseline['acertada'].mean()
        resultados['baseline'] = ex_base
        print(f'Baseline EX: {ex_base:.4f}')
    
    if config_a is not None:
        ex_a = config_a['acertada'].mean()
        resultados['config_a'] = ex_a
        print(f'Config A EX: {ex_a:.4f}')
        if baseline is not None:
            mejora = ex_a - ex_base
            print(f'Mejora sobre baseline: {mejora:+.4f}')
    
    if config_b is not None:
        ex_b = config_b['acertada'].mean()
        resultados['config_b'] = ex_b
        print(f'Config B EX: {ex_b:.4f}')
        if baseline is not None:
            mejora = ex_b - ex_base
            print(f'Mejora sobre baseline: {mejora:+.4f}')
    
    return resultados

## 4. Análisis de errores por categoría

In [ ]:
def clasificar_error(sql_generada, sql_referencia, error_msg=None):
    """Clasifica el tipo de error en la SQL generada."""
    if not sql_generada:
        return 'sin_respuesta'
    
    # Categorías de error
    categorias = {
        'sintaxis': False,
        'columna_inexistente': False,
        'tabla_inexistente': False,
        'join': False,
        'agregacion': False,
        'filtro': False,
        'resultado_distinto': False
    }
    
    if error_msg:
        error_lower = error_msg.lower()
        if 'syntax' in error_lower or 'parse' in error_lower:
            categorias['sintaxis'] = True
        elif 'unknown column' in error_lower or 'column' in error_lower:
            categorias['columna_inexistente'] = True
        elif "doesn't exist" in error_lower or 'table' in error_lower:
            categorias['tabla_inexistente'] = True
    else:
        # La query se ejecutó pero dio resultado diferente
        categorias['resultado_distinto'] = True
        
        # Analizar diferencias
        sql_gen_upper = sql_generada.upper()
        sql_ref_upper = sql_referencia.upper()
        
        if 'JOIN' in sql_ref_upper and 'JOIN' not in sql_gen_upper:
            categorias['join'] = True
        if any(kw in sql_ref_upper for kw in ['COUNT', 'SUM', 'AVG', 'MAX', 'MIN', 'GROUP BY']):
            if not any(kw in sql_gen_upper for kw in ['COUNT', 'SUM', 'AVG', 'MAX', 'MIN', 'GROUP BY']):
                categorias['agregacion'] = True
        if 'WHERE' in sql_ref_upper and 'WHERE' not in sql_gen_upper:
            categorias['filtro'] = True
    
    # Retornar la categoría principal
    for cat, es_error in categorias.items():
        if es_error:
            return cat
    return 'desconocido'

print('Función de clasificación de errores lista')

In [ ]:
def analizar_errores(resultados_df):
    """Analiza los errores del modelo."""
    errores = resultados_df[~resultados_df['acertada']].copy()
    
    errores['categoria'] = errores.apply(
        lambda row: clasificar_error(
            row.get('sql_generada', ''),
            row.get('sql_referencia', ''),
            None
        ),
        axis=1
    )
    
    conteo = errores['categoria'].value_counts()
    print('Errores por categoría:')
    for cat, count in conteo.items():
        pct = count / len(errores) * 100
        print(f'  {cat}: {count} ({pct:.1f}%)')
    
    return errores

## 5. Experimento de inferencia

Self-correction: usar `ejecutar()` para corregir queries incorrectas.

In [ ]:
def experimento_inferencia(esquema, preguntas, ejecutar_fn):
    """
    Experimento de inferencia con self-correction.
    
    Estrategia:
    1. Generar SQL sin ejecución
    2. Ejecutar la SQL
    3. Si falla, re-generar con el error como contexto
    4. Hasta 3 intentos
    """
    print('=== Experimento de Inferencia: Self-Correction ===')
    
    resultados_sin_corr = []
    resultados_con_corr = []
    
    for item in preguntas:
        # Sin corrección
        sql_sin = generar_sql(esquema, item['pregunta'])
        resultados_sin_corr.append(sql_sin)
        
        # Con corrección
        sql_con = generar_sql(esquema, item['pregunta'], ejecutar=ejecutar_fn)
        resultados_con_corr.append(sql_con)
    
    return resultados_sin_corr, resultados_con_corr

## 6. Predicción de EX

In [ ]:
def predecir_ex(ex_validacion_propia, ex_baseline, num_tablas_eval=7, num_preguntas=35):
    """
    Predice el EX en la evaluación oculta.
    
    Justificación:
    - El dataset de validación propia tiene 2 esquemas conocidos
    - La evaluación oculta tendrá esquemas desconocidos de 5-10 tablas
    - Esperamos una caída del 10-20% por generalización
    - Pero las preguntas siguen la misma distribución de niveles
    """
    # Factor de generalización
    factor_generalizacion = 0.85  # 15% de caída esperada
    
    ex_predicho = ex_validacion_propia * factor_generalizacion
    
    print('=== Predicción de EX en evaluación oculta ===')
    print(f'EX en validación propia: {ex_validacion_propia:.4f}')
    print(f'Factor de generalización: {factor_generalizacion}')
    print(f'EX predicho: {ex_predicho:.4f}')
    print(f'')
    print(f'Justificación:')
    print(f'- El modelo fue entrenado en 10 esquemas propios + Spider/BIRD')
    print(f'- La evaluación oculta tendrá esquemas no vistos')
    print(f'- Esperamos ~15% de caída por generalización a nuevos esquemas')
    print(f'- El modelo maneja bien patrones SQL estándar')
    print(f'- Las preguntas difíciles (ventanas, subconsultas) bajarán el EX')
    
    return ex_predicho